# Sentiment Analysis using IMDB Reviews Dataset

## Importing Libraries

In [1]:
import pandas as pd

## Loading the Dataset

In [2]:
import kagglehub

dataset_path = kagglehub.dataset_download('lakshmi25npathi/imdb-dataset-of-50k-movie-reviews')
csv_file_path = f'{dataset_path}/IMDB Dataset.csv'

print(f"Dataset downloaded to: {dataset_path}")
print(f"CSV file path: {csv_file_path}")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Dataset downloaded to: /kaggle/input/imdb-dataset-of-50k-movie-reviews
CSV file path: /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [3]:
df = pd.read_csv(csv_file_path)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Preprocessing the Data

### Lowercasing the Text

In [5]:
df['review'] = df['review'].str.lower()

In [6]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


In [7]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


### Removing HTML Tags

In [8]:
# removing HTML tags
from bs4 import BeautifulSoup

def remove_html_tags(text):
    return BeautifulSoup(text, "html.parser").get_text()

In [9]:
df['review'] = df['review'].apply(remove_html_tags)

In [10]:
df['review']

,review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production. the filming tec...
2,i thought this was a wonderful way to spend ti...
3,basically there's a family where a little boy ...
4,"petter mattei's ""love in the time of money"" is..."
...,...
49995,i thought this movie did a down right good job...
49996,"bad plot, bad dialogue, bad acting, idiotic di..."
49997,i am a catholic taught in parochial elementary...
49998,i'm going to have to disagree with the previou...


In [11]:
# Display first 5 reviews after removing HTML tags
for i in range(5):
    print(f"Review {i+1}:\n{df['review'][i]}\n")

Review 1:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other

In [12]:
from bs4 import BeautifulSoup

html = 'Check out "https://google.com">this search engine</a> now.'
soup = BeautifulSoup(html, "html.parser")

# This removes the <a> tags but leaves "this search engine"
clean_text = soup.get_text()

print(clean_text) 
# Output: Check out this search engine now.

Check out "https://google.com">this search engine now.


### Search url in Text

In [13]:
def search_url(text):
    pattern = r'http\S+|www\S+|https\S+'
    match = re.search(pattern, text)
    if match:
        print(re.search(pattern, text).group())
    else:
        print('no match found')
    

### Remove URLs

In [14]:
# remove urls
import re
def remove_urls(text):
    url_pattern = re.compile(r'http\S+|www\S+|https\S+', re.IGNORECASE)
    return url_pattern.sub(r'', text)

text = 'Check out https://google.com and http://site.org now.'
cleaned_text = remove_urls(text)
print(cleaned_text)  

Check out  and  now.


In [15]:
df['review'] = df['review'].apply(remove_urls)

### Remove Punctuation

In [16]:
import string
import time

In [17]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [18]:
df.shape

(50000, 2)

In [19]:
def remove_punct(text):
    translator = str.maketrans('','', string.punctuation)
    text = text.translate(translator)
    return text

In [20]:
df['review'] = df['review'].apply(remove_punct)

### Converting Slang Words to Normal Words

In [21]:
slang_text = pd.read_csv('slang.txt', sep='=',names=['Abbreviation', 'Meaning'], header=None)

In [22]:
slang_text

,Abbreviation,Meaning
0,A3,"Anytime, Anywhere, Anyplace"
1,ADIH,Another Day In Hell
2,AFK,Away From Keyboard
3,AFAIK,As Far As I Know
4,ASAP,As Soon As Possible
...,...,...
99,WTG,Way To Go!
100,WUF,Where Are You From?
101,WYD,What You Doing?
102,WYWH,Wish You Were Here


In [23]:
slang_text_dict = slang_text.set_index('Abbreviation')['Meaning'].to_dict()

In [24]:
slang_text_dict

{'A3': 'Anytime, Anywhere, Anyplace',
 'ADIH': 'Another Day In Hell',
 'AFK': 'Away From Keyboard',
 'AFAIK': 'As Far As I Know',
 'ASAP': 'As Soon As Possible',
 'ASL': 'Age, Sex, Location',
 'ATK': 'At The Keyboard',
 'ATM': 'At The Moment',
 'BAE': 'Before Anyone Else',
 'BAK': 'Back At Keyboard',
 'BBL': 'Be Back Later',
 'BBS': 'Be Back Soon',
 'BFN': 'Bye For Now',
 'B4N': 'Bye For Now',
 'BRB': 'Be Right Back',
 'BRUH': 'Bro',
 'BRT': 'Be Right There',
 'BSAAW': 'Big Smile And A Wink',
 'BTW': 'By The Way',
 'BWL': 'Bursting With Laughter',
 'CSL': 'Can’t Stop Laughing',
 'CU': 'See You',
 'CUL8R': 'See You Later',
 'CYA': 'See You',
 'DM': 'Direct Message',
 'FAQ': 'Frequently Asked Questions',
 'FC': 'Fingers Crossed',
 'FIMH': 'Forever In My Heart',
 'FOMO': 'Fear Of Missing Out',
 'FR': 'For Real',
 'FWIW': "For What It's Worth",
 'FYP': 'For You Page',
 'FYI': 'For Your Information',
 'G9': 'Genius',
 'GAL': 'Get A Life',
 'GG': 'Good Game',
 'GMTA': 'Great Minds Think Alik

In [25]:
'ATM' in 'my nam eis ATM'

True

In [26]:
def slang_to_text(text):
    text_words = text.split(' ')
    new_text = []
    for word in text_words:
        if word.upper() in slang_text_dict:
            new_text.append(slang_text_dict[word.upper()])
        else:
            new_text.append(word)

    return ' '.join(new_text)

In [27]:
slang_to_text('my name is ATM')

'my name is At The Moment'

### Correcting Spelling Mistakes

In [28]:
from textblob import TextBlob

In [29]:
incorrect_text = 'ai stand for artifcal inreligence'
corrrect_text = TextBlob(incorrect_text).correct().string

In [30]:
corrrect_text

'ai stand for artificial intelligence'

### Removing Stop Words

In [31]:
import nltk

In [32]:
# Essential for stopword removal
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
from nltk.corpus import stopwords

In [34]:
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [35]:
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    cleaned_text = [word for word in text.split() if word.lower() not in stop_words]
    return ' '.join(cleaned_text)

In [36]:
df['review'] = df['review'].apply(remove_stopwords)

### Removing Emojis

In [38]:
import sys
import subprocess
import importlib.util

package_name = 'emoji'

if importlib.util.find_spec(package_name) is None:
    print(f"Installing {package_name}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
else:
    print(f"{package_name} is already installed. Skipping.")

Installing emoji...


In [39]:
import emoji

In [40]:
emoji.replace_emoji('I am so excited for the concert tonight! 🎤🎸✨', replace='')

'I am so excited for the concert tonight! '

In [41]:
emoji.demojize('I am so excited for the concert tonight! 🎤🎸✨')

'I am so excited for the concert tonight! :microphone::guitar::sparkles:'

In [42]:
df['review'] = df['review'].apply(
    lambda text:emoji.replace_emoji(text, replace='')
)

In [43]:
emoji_pattern = re.compile(r"[\U00010000-\U0010ffff]", flags=re.UNICODE)

df['review'] = df['review'].astype(str).str.replace(emoji_pattern, '', regex=True)

In [47]:
text = 'I am so excited for the concert tonight! 🎤🎸✨'
re.sub(emoji_pattern,'', text)

'I am so excited for the concert tonight! ✨'